# tribe-bench: Setup & Smoke Test (the GO / NO-GO gate)

Run on **Kaggle** with a **GPU T4** accelerator and **Internet = ON** (Settings panel).
Internet is mandatory: pip installs, the HuggingFace checkpoint (~1 GB), the LLaMA-3.2 gated
download, the WhisperX model (~3 GB), and the sample video all need it.

One run answers the three questions the whole project waits on:

1. **Does TRIBE v2 install on Kaggle?**  (G016)
2. **Does it fit a 16 GB T4?**  peak VRAM, incl. per-extractor breakdown  (G005)
3. **Does modality ablation actually change the output?**  (G018 — BrainLens's whole mechanic)

This notebook is built to *learn*, not just to succeed: every phase is wrapped so a failure
in one step still prints diagnostics and lets the independent later cells run. Read the
verdict printed by the last cell.

---
### Verified-from-source facts this notebook relies on
- `TribeModel.from_pretrained(checkpoint_dir, checkpoint_name='best.ckpt', cache_folder=None, cluster=None, device='auto', config_update=None)` — **`device` and `cache_folder` ARE accepted** (demo_utils.py:150-159). Our `load_model(device=..., cache_folder=...)` call is correct.
- `get_events_dataframe(video_path=...)` does **NOT** run the neural encoders. It only: extracts the audio track, chunks, and runs **WhisperX ASR** (via `uvx whisperx --model large-v3`, eventstransforms.py:111-137) to get word timings, then text annotation. So `uv` must be installed and Internet on, or every video/audio/text input fails at the ASR step.
- The encoders (V-JEPA2 / Wav2Vec-BERT / LLaMA-3.2) run **inside `model.predict()`** — specifically inside `Data.get_loaders()`, which reads `self.features_to_use` **at call time** to decide which extractors to `prepare()` (main.py:160-217). They load, cache to disk, and free VRAM via `_free_extractor_model()` sequentially.
- **Ablation mechanism (G018) is genuinely valid.** Because extraction is decided at predict-time from `model.data.features_to_use`, mutating that list *before* `predict()` controls which extractors run. A modality that is not extracted is **absent from `batch.data`, and the brain model substitutes zeros for it** (model.py:188-192; `feature_dims` was fixed at build time to all trained modalities). So masking a modality changes the output. `_find_features_to_use()` path #1 `model.data.features_to_use` matches a real pydantic field (main.py:98) — the path is known.
  - Note: `features_to_mask` (the config field) has **no effect at inference** — it is only read at trainer build time (main.py:487). Our wrapper correctly uses `features_to_use` mutation instead.

## Phase 1 - Install

TRIBE v2 is a GitHub repo (not on PyPI). `pip install -e .` pulls its pinned deps
(`neuralset==0.0.2`, `neuraltrain==0.0.2`, `torch>=2.5.1,<2.7`, `numpy==2.2.6`, `x_transformers==1.27.20`, ...)
from PyPI automatically (verified in tribev2/pyproject.toml). `exca` arrives transitively.

**Two things that will bite and are the point of this test:**
- `neuralset==0.0.2` requires **Python >= 3.12** (G016). Kaggle images are often on 3.11 — if so the resolver will refuse `neuralset`. If you see that error, that is the finding: note it and stop (a 3.12 Kaggle image or a uv/conda env is required).
- The strict `numpy==2.2.6` / `torch` range pins will churn Kaggle's preinstalled stack and may reinstall torch. Watch for a CUDA/torch mismatch in the env-check cell below.

**`uv` is installed here on purpose** — WhisperX is invoked as `uvx whisperx`, so the `uvx` binary must exist or `get_events_dataframe` cannot transcribe.

In [ ]:
import sys, subprocess, shutil, importlib
print('Python:', sys.version)

# ROOT-CAUSE GUARD (2026-07-22, v2): Phase 6 failed with
#   ModuleNotFoundError: No module named 'tribev2.demo_utils'
# i.e. `import tribev2` resolved to a package WITHOUT demo_utils. Two Kaggle causes,
# both fixed below: (1) a bare `tribev2` dir on cwd (/kaggle/working is on sys.path)
# resolves as an empty namespace package; (2) an editable-install .pth only activates
# on a FRESH interpreter, so mid-session `import tribev2` can miss tribev2_src entirely.
# Fix: clone to a non-colliding path AND put that path on sys.path EXPLICITLY (don't
# trust the .pth), then hard-assert the resolved file before continuing.
SRC = '/kaggle/working/tribev2_src'
shutil.rmtree('/kaggle/working/tribev2', ignore_errors=True)
shutil.rmtree(SRC, ignore_errors=True)
for _m in [m for m in list(sys.modules) if m == 'tribev2' or m.startswith('tribev2.')]:
    sys.modules.pop(_m, None)

# uv/uvx binary - REQUIRED by WhisperX ASR inside get_events_dataframe (eventstransforms.py:111).
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=False)

# TRIBE v2 (public GitHub repo). Clone to a NON-colliding path (tribev2_src, NOT tribev2).
subprocess.run(f'git clone --depth 1 https://github.com/facebookresearch/tribev2.git {SRC}',
               shell=True, check=False)
# editable install; deps (neuralset==0.0.2, neuraltrain==0.0.2, exca, torch, ...) resolve from PyPI.
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', SRC],
                   capture_output=True, text=True)
print('tribev2 install rc =', r.returncode)
if r.returncode != 0:
    print('--- STDERR (tail) ---')
    print(r.stderr[-3000:])
    print('>>> If this mentions requires-python / neuralset>=3.12, Kaggle is on Python <3.12 (G016).')

# Resolve tribev2 by DIRECTORY, not via the editable .pth (which needs a fresh interpreter).
# SRC holds the package at SRC/tribev2/, so SRC on sys.path makes `import tribev2` authoritative.
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()
print('install step done - inspect above for resolver/version errors')

# FAIL-FAST (~2 min in, not 20) - and it RAISES, so Run All stops here instead of
# cascading to a confusing Phase 6. `tribev2` must resolve INTO tribev2_src.
import tribev2
where = getattr(tribev2, '__file__', None) or list(getattr(tribev2, '__path__', []))
print('tribev2 resolves to:', where)
assert where and 'tribev2_src' in str(where), (
    f'SHADOWED: tribev2 resolved to {where}, not {SRC}. '
    'A stray tribev2 dir is winning on sys.path - restart the kernel (Factory reset) and re-run.')
importlib.import_module('tribev2.demo_utils')
print('OK: tribev2.demo_utils imports -> Phase 6 model load will work.')


### tribe-bench itself (now a PUBLIC GitHub repo)

The cell below clones it directly with `git clone` — **no token needed** now that the repo
is public. It also falls back to a Kaggle-dataset mount or a `GH_TOKEN` secret in case you
make the repo private again later.

In [ ]:
import os, sys, subprocess, glob
from pathlib import Path

TB_PATH = None
# Option 0 (primary): public repo - plain clone, no token needed.
subprocess.run('git clone --depth 1 https://github.com/codesbydevesh/tribe-bench.git /kaggle/working/tribe-bench',
               shell=True, check=False)
if Path('/kaggle/working/tribe-bench/tribe_tools/model.py').is_file():
    TB_PATH = '/kaggle/working/tribe-bench'

# Option 1 (fallback): attached Kaggle dataset.
if TB_PATH is None:
    for cand in glob.glob('/kaggle/input/*') + glob.glob('/kaggle/input/*/*'):
        if Path(cand, 'tribe_tools', 'model.py').is_file():
            TB_PATH = cand; break

# Option 2 (fallback): token clone (only needed if repo is private again).
if TB_PATH is None:
    tok = os.environ.get('GH_TOKEN', '')
    if not tok:
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret('GH_TOKEN')
        except Exception:
            tok = ''
    if tok:
        url = f'https://{tok}@github.com/codesbydevesh/tribe-bench.git'
        subprocess.run(f'git clone --depth 1 {url} /kaggle/working/tribe-bench', shell=True, check=False)
        if Path('/kaggle/working/tribe-bench/tribe_tools/model.py').is_file():
            TB_PATH = '/kaggle/working/tribe-bench'

if TB_PATH:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', TB_PATH], check=False)
    if TB_PATH not in sys.path:
        sys.path.insert(0, TB_PATH)
    print('tribe-bench found at:', TB_PATH)
else:
    print('tribe-bench NOT available - check the clone error above (repo should be public).')
    print('Phases 1-3 + raw TRIBE checks still run; Phase 4 wrapper import will be skipped.')

## Phase 2 - Environment check

In [ ]:
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB')
else:
    print('NO GPU - enable a T4 accelerator in Kaggle Settings, then re-run.')

import shutil
print('ffmpeg on PATH:', bool(shutil.which('ffmpeg')))
print('uvx on PATH   :', bool(shutil.which('uvx')), '(required for WhisperX ASR)')
try:
    import numpy as _np; print('numpy:', _np.__version__)
except Exception as e:
    print('numpy import failed:', e)

## Phase 3 - HuggingFace login (required for gated LLaMA-3.2)

The text extractor pulls **LLaMA-3.2-3B**, which is gated. Request access once at
huggingface.co/meta-llama/Llama-3.2-3B, then add a Kaggle secret `HF_TOKEN` (or set the env var).
Without it, the model checkpoint download works but the *text* extractor will fail during predict.

In [ ]:
import os
hf_ok = False
try:
    from huggingface_hub import login
    token = os.environ.get('HF_TOKEN', '')
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret('HF_TOKEN')
        except Exception:
            token = ''
    if token:
        login(token=token)
        os.environ['HF_TOKEN'] = token
        hf_ok = True
        print('HF login OK')
    else:
        print('No HF_TOKEN. Set a Kaggle secret HF_TOKEN. The text (LLaMA) extractor will fail without it.')
except Exception as e:
    print('HF login error:', type(e).__name__, e)

## Phase 4 - Import the wrapper

Only `tribe_tools.model` is required for the gate. The other modules are imported defensively
so a missing/heavy optional dep (e.g. plotting libs in `viz`) does not abort the run.

In [ ]:
WRAPPER_OK = False
try:
    from tribe_tools.model import load_model, predict_single, MODALITY_MASKS, _find_features_to_use
    WRAPPER_OK = True
    print('tribe_tools.model imports: OK')
    print('MODALITY_MASKS:', MODALITY_MASKS)
except Exception as e:
    print('WRAPPER IMPORT FAILED:', type(e).__name__, e)
    print('Fix Phase-1 tribe-bench availability before the ablation phases can run.')

# Optional / non-gating modules - never abort on these.
for _m in ['tribe_tools.atlas', 'tribe_tools.cache', 'tribe_tools.viz',
           'tribe_tools.video_utils', 'neurocheck.claims']:
    try:
        __import__(_m); print('  optional import OK:', _m)
    except Exception as e:
        print('  optional import skipped:', _m, '->', type(e).__name__, e)

## Phase 5 - Make a real test clip (with speech)

A synthetic `testsrc` + 440 Hz tone has **no speech**, so WhisperX would return an empty
transcript and the *text* modality would be zero in every pass — making the text half of the
ablation meaningless. We use Meta's own demo stimulus (the public-domain **Sintel** trailer),
trimmed to the first 30 s to bound cost while keeping dialogue/narration. This gives a genuine
video-vs-audio-vs-text contrast.

In [ ]:
from pathlib import Path
import subprocess
CACHE = Path('/kaggle/working/cache'); CACHE.mkdir(parents=True, exist_ok=True)
raw = CACHE / 'sintel_full.mp4'
video_path = Path('/kaggle/working/test_clip.mp4')
url = 'https://download.blender.org/durian/trailer/sintel_trailer-480p.mp4'
try:
    from tribev2.demo_utils import download_file
    if not raw.exists() or raw.stat().st_size == 0:
        download_file(url, raw)
    # trim first 30s (stream-copy; fall back to re-encode if the copy trim is empty)
    subprocess.run(['ffmpeg', '-y', '-i', str(raw), '-t', '30', '-c', 'copy', str(video_path)],
                   check=False, capture_output=True)
    if (not video_path.exists()) or video_path.stat().st_size == 0:
        subprocess.run(['ffmpeg', '-y', '-i', str(raw), '-t', '30', str(video_path)],
                       check=False, capture_output=True)
    print('clip exists:', video_path.exists(),
          '| size:', video_path.stat().st_size if video_path.exists() else 0, 'bytes')
except Exception as e:
    print('STIMULUS PREP FAILED:', type(e).__name__, e)
    print('Check Internet is ON and tribev2 installed.')

## Phase 6 - Load the model (timed)

In [ ]:
import time
tribe = None; load_time = None
if WRAPPER_OK:
    try:
        t0 = time.time()
        tribe = load_model(device='cuda', cache_folder=CACHE)
        load_time = time.time() - t0
        print(f'Model loaded in {load_time:.1f}s')
    except Exception as e:
        import traceback; traceback.print_exc()
        print('MODEL LOAD FAILED:', type(e).__name__, e)
else:
    print('Skipped: wrapper not importable.')

## Phase 7 - Full prediction + peak VRAM  (does it fit a T4? - G005)

We monkeypatch `tribev2.main._free_extractor_model` to record `max_memory_allocated`
**at each extractor's free point** and reset the peak counter afterward. Because extractors
load/free sequentially with the (~1 GB) brain model resident throughout, each recorded value
is that stage's peak with the brain model included — the real G005 per-encoder number. The
final peak after the last free is the brain-model forward pass. Overall peak = the max of all.

The patch is fully guarded and always restored; if it misbehaves we still get a total peak.

In [ ]:
import torch, time
preds = None; segments = None; infer_time = None
per_extractor_vram = {}; peak_vram = None; forward_peak = None

if tribe is not None and video_path.exists():
    import tribev2.main as tmain
    _orig_free = tmain._free_extractor_model
    _stage = {'i': 0}
    def _patched_free(extractor):
        try:
            if torch.cuda.is_available():
                torch.cuda.synchronize()
                pk = torch.cuda.max_memory_allocated() / 1e9
                label = type(extractor).__name__
                data_obj = getattr(tribe, 'data', None)
                for mod in ('text', 'audio', 'video', 'image'):
                    if getattr(data_obj, f'{mod}_feature', None) is extractor:
                        label = mod; break
                per_extractor_vram[f"{_stage['i']}:{label}"] = round(pk, 2)
                _stage['i'] += 1
        except Exception as _e:
            per_extractor_vram[f"stage{_stage['i']}_err"] = str(_e); _stage['i'] += 1
        _orig_free(extractor)
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
    try:
        tmain._free_extractor_model = _patched_free
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        t0 = time.time()
        preds, segments = predict_single(tribe, video_path)  # full: all modalities
        infer_time = time.time() - t0
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            forward_peak = torch.cuda.max_memory_allocated() / 1e9  # brain-model forward
        peak_vram = max([forward_peak or 0] + list(per_extractor_vram.values()) or [0])
        print('=== FULL PREDICTION ===')
        print('shape:', preds.shape, '| dtype:', preds.dtype)
        print(f'range: [{preds.min():.4f}, {preds.max():.4f}]  mean: {preds.mean():.4f}')
        print('segments kept:', len(segments))
        print(f'inference time: {infer_time:.1f}s')
        print('per-extractor peak VRAM (GB, incl. resident brain model):')
        for k, v in per_extractor_vram.items():
            print(f'    {k:<16} {v} GB')
        print(f'brain-model forward peak: {forward_peak:.2f} GB' if forward_peak else 'forward peak: n/a')
        print(f'OVERALL PEAK VRAM: {peak_vram:.2f} GB   (T4 usable ~15 GB)')
    except Exception as e:
        import traceback; traceback.print_exc()
        print('FULL PREDICTION FAILED:', type(e).__name__, e)
        print('If this is a WhisperX/uvx error, ASR is the blocker - see the Phase-8b fallback cell.')
    finally:
        tmain._free_extractor_model = _orig_free
else:
    print('Skipped: model not loaded or clip missing.')

## Phase 8 - THE decisive test: does modality ablation change the output?  (G018)

First we print where `features_to_use` actually lives (must resolve to `model.data.features_to_use`),
and the real modality list baked into the pretrained config. Then we run a **video-only** pass
(audio + text masked) and compare it to the full pass. **They must differ.** If they are
identical, BrainLens's core mechanic is dead as built -> pivot to NeuroCheck-only.

In [ ]:
import numpy as np
if tribe is not None:
    print('has .data:', hasattr(tribe, 'data'))
    loc = _find_features_to_use(tribe)
    if loc is None:
        print('_find_features_to_use -> None  (unexpected: model.data.features_to_use should match)')
    else:
        parent, attr = loc
        print('_find_features_to_use ->', type(parent).__name__, '.', attr,
              '=', list(getattr(parent, attr)))
        print('features_to_mask (config, inference-inert):', getattr(tribe.data, 'features_to_mask', 'n/a'))
else:
    print('Skipped: model not loaded.')

In [ ]:
import numpy as np
verdict = None; maxdiff = None
if tribe is not None and preds is not None and video_path.exists():
    try:
        preds_vid, _seg = predict_single(tribe, video_path, features_to_mask=['audio', 'text'])
        if preds_vid.shape == preds.shape:
            identical = np.allclose(preds_vid, preds)
            maxdiff = float(np.abs(preds_vid - preds).max())
            verdict = 'DEAD (identical output)' if identical else f'WORKS (max abs diff {maxdiff:.5f})'
        else:
            verdict = f'WORKS (shape changed: full {preds.shape} vs video-only {preds_vid.shape})'
    except RuntimeError as e:
        verdict = f'PATH NOT FOUND (G018): {e}'
    except Exception as e:
        import traceback; traceback.print_exc()
        verdict = f'ERROR: {type(e).__name__}: {e}'
else:
    verdict = 'NOT RUN (full prediction unavailable)'
print('ABLATION VERDICT:', verdict)

## Phase 8b - Fallback ablation WITHOUT WhisperX  (run only if Phase 7/8 failed at ASR)

If WhisperX/uvx is the blocker, we can still answer VRAM (G005) and a *video-vs-audio*
ablation (G018, minus text) by building events with `audio_only=True`, which skips
`ExtractWordsFromAudio` entirely (demo_utils.py:80-91). This proves the masking mechanism
end-to-end even when ASR is unavailable. Text is not exercised in this fallback.

In [ ]:
import numpy as np, torch, time
run_fallback = (tribe is not None) and (preds is None) and video_path.exists()
if run_fallback:
    try:
        import pandas as pd
        from tribev2.demo_utils import get_audio_and_text_events
        ev = {'type': 'Video', 'filepath': str(video_path), 'start': 0,
              'timeline': 'default', 'subject': 'default'}
        df_noasr = get_audio_and_text_events(pd.DataFrame([ev]), audio_only=True)
        print('events built (audio_only, no ASR). rows:', len(df_noasr))
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        t0 = time.time()
        preds_fb, seg_fb = tribe.predict(events=df_noasr)
        fb_time = time.time() - t0
        fb_peak = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else None
        print('FALLBACK FULL:', preds_fb.shape, '| time %.1fs' % fb_time,
              '| peak %.2f GB' % fb_peak if fb_peak else '')
        # video-only ablation via features_to_use mutation (mirrors predict_single logic)
        loc = _find_features_to_use(tribe)
        parent, attr = loc
        original = list(getattr(parent, attr))
        keep = [f for f in original if f not in ('audio', 'text')]
        setattr(parent, attr, keep)
        try:
            preds_fb_vid, _ = tribe.predict(events=df_noasr)
        finally:
            setattr(parent, attr, original)
        if preds_fb_vid.shape == preds_fb.shape:
            md_ = float(np.abs(preds_fb_vid - preds_fb).max())
            print('FALLBACK ABLATION:', 'DEAD (identical)' if np.allclose(preds_fb_vid, preds_fb)
                  else f'WORKS (max abs diff {md_:.5f})')
        else:
            print('FALLBACK ABLATION: WORKS (shape changed)', preds_fb.shape, preds_fb_vid.shape)
    except Exception as e:
        import traceback; traceback.print_exc()
        print('FALLBACK FAILED:', type(e).__name__, e)
else:
    print('Fallback not needed (primary path ran) or model unavailable.')

## Phase 9 - Save + record

Saves outputs to `/kaggle/working` (download before the 12h session ends), and prints the
block to paste into `ops/source-of-truth.md` (G005) and `ops/knowledge-gaps.md` (G005/G018).

In [ ]:
import numpy as np, sys
if preds is not None:
    np.save('/kaggle/working/smoke_full.npy', preds)
print('=== RECORD IN ops/source-of-truth.md + knowledge-gaps.md ===')
print('python              :', sys.version.split()[0])
print('output shape        :', None if preds is None else preds.shape)
print('n_vertices          :', None if preds is None else preds.shape[1])
print('load time (s)       :', None if load_time is None else round(load_time, 1))
print('inference time (s)  :', None if infer_time is None else round(infer_time, 1))
print('per-extractor VRAM  :', per_extractor_vram, '  -> G005')
print('brain fwd peak (GB) :', None if forward_peak is None else round(forward_peak, 2))
print('OVERALL peak (GB)   :', None if peak_vram is None else round(peak_vram, 2), '  -> G005')
print('ablation (G018)     :', verdict)
print('=============================================================')